In [2]:
import pandas as pd
import os

RAW_INPUT_PATH = "../data/raw/remoteok_raw.csv"
CLEAN_OUTPUT_PATH = "../data/cleaned/remoteok_jobs_cleaned.csv"


job_type_keywords = {
    "full": "Full-Time",
    "contract": "Contract",
    "part": "Part-Time",
    "freelance": "Freelance",
    "intern": "Internship",
    "temp": "Temporary"
}


def infer_job_type(raw_type, tags):
    if pd.notna(raw_type) and raw_type != "":
        return str(raw_type).title()

    for tag in tags:
        tag_l = tag.lower()
        for k, v in job_type_keywords.items():
            if k in tag_l:
                return v

    return "Not Specified"


def load_raw_dataset():
    print("Loading raw dataset...")
    df = pd.read_csv(RAW_INPUT_PATH)
    print(f"Rows loaded: {len(df)}")
    return df


def clean_dataset(df):
    print("Cleaning data...")

    # Split tags for inference
    df["Tag_List"] = df["Job Tags / Skills"].fillna("N/A").astype(str).str.split(", ")

    # Infer + normalize job type
    df["Job Type"] = df.apply(
        lambda row: infer_job_type(row["Job Type (Raw)"], row["Tag_List"]),
        axis=1
    )

    # Remove duplicates
    df.drop_duplicates(
        subset=["Job Title", "Company Name", "Job URL"],
        inplace=True
    )

    # Clean text fields
    for col in ["Job Title", "Company Name", "Location", "Job Type"]:
        df[col] = df[col].astype(str).str.strip()

    # Replace missing location
    df["Location"] = df["Location"].replace({"": "Worldwide", "nan": "Worldwide"})

    # Drop helper column
    df.drop(columns=["Tag_List"], inplace=True)

    print("Cleaning complete.")
    print(f"Final rows: {len(df)}")

    return df


def save_cleaned_dataset(df):
    os.makedirs("../data/cleaned", exist_ok=True)
    df.to_csv(CLEAN_OUTPUT_PATH, index=False)
    print(f"Saved cleaned dataset to: {CLEAN_OUTPUT_PATH}")


def main():
    df_raw = load_raw_dataset()
    df_clean = clean_dataset(df_raw)
    save_cleaned_dataset(df_clean)


if __name__ == "__main__":
    main()


Loading raw dataset...
Rows loaded: 100
Cleaning data...
Cleaning complete.
Final rows: 100
Saved cleaned dataset to: ../data/cleaned/remoteok_jobs_cleaned.csv
